# K-ABENA — Niveau 1 : Régression Linéaire
**Tutoriel notebook** — copiez-collez dans Jupyter, Colab ou Kaggle.

## Principe en 1 phrase
> K-ABENA exclut les observations dont l'erreur est trop petite pour être informative,
> et concentre le gradient sur les erreurs qui comptent vraiment.

**Coût d'adoption : +2 lignes dans votre code existant.**

In [ ]:
# Installation
# !pip install kabena-ml scikit-learn matplotlib -q

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# ── Données ──────────────────────────────────────────────────────────────
data    = fetch_california_housing()
X       = StandardScaler().fit_transform(data.data)
y       = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Dataset : {X_train.shape[0]} train, {X_test.shape[0]} test, {X.shape[1]} features")

## ① Régression Ridge standard (référence)

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
mse_std = mean_squared_error(y_test, ridge.predict(X_test))
print(f"MSE Ridge standard : {mse_std:.4f}")

## ② Ridge + K-ABENA
**Seul changement :** remplacer `Ridge()` par `KabenaWrapper(Ridge(), K=...)` — API identique.

In [ ]:
from kabena_ml import calibrate_K
from kabena_ml.integrations.sklearn_wrapper import KabenaWrapper

# Étape 1 — Calibrer K automatiquement depuis l'époque 1
errors_ep1 = np.abs(y_train - ridge.predict(X_train))
K_auto     = calibrate_K(errors_ep1, target_pct=0.10)
print(f"K calibré automatiquement : {K_auto:.4f}")

# Étape 2 — Remplacer Ridge par KabenaWrapper(Ridge(), K=K_auto)
ka_ridge = KabenaWrapper(
    Ridge(alpha=1.0),   # ← votre estimateur habituel
    K      = K_auto,    # ← seuil auto
    N      = 0.0,       # ← toutes les mineures exclues
    epochs = 200,
    task   = "regression",
    verbose= True,
)
ka_ridge.fit(X_train, y_train)

mse_ka = mean_squared_error(y_test, ka_ridge.predict(X_test))
print(f"\nMSE Ridge standard : {mse_std:.4f}")
print(f"MSE Ridge K-ABENA  : {mse_ka:.4f}")
print(f"Gain computationnel : {ka_ridge.stats_['mean_gain_pct']:.1f}%")

## ③ Exploration de la grille K × N

In [ ]:
from kabena_ml.utils.logger import benchmark_KN

results = benchmark_KN(
    X_train, y_train, X_test, y_test,
    estimator_fn = lambda: Ridge(alpha=1.0),
    K_range      = [0.05, 0.10, 0.20, 0.30],
    N_range      = [0.0, 0.3, 0.6, 1.0],
    epochs       = 100,
    scoring      = "mse",
)
print("\nMeilleur résultat :", results.best_params())
results.plot_heatmap()